In [ ]:
# hide
import numpy as np
import pyquist as pq

In [ ]:
# collapse
def rice_bits(u: np.ndarray, k: int) -> int:
    """Total length of the Rice codes of non-negative integers u: a unary quotient
    (u >> k ones plus a zero), then a k-bit remainder."""
    return int(np.sum((u >> k) + 1 + k))


def zigzag(e: np.ndarray) -> np.ndarray:
    return np.where(e >= 0, 2 * e, -2 * e - 1)

In [ ]:
def encode_block(x: np.ndarray, K: int = 8, shift: int = 12) -> tuple[np.ndarray, np.ndarray]:
    """Fit LPC coefficients to a block of integer samples, quantize them to integers
    (in units of 2^-shift), and return them along with the integer residual."""
    N = len(x)
    X = np.stack([x[K - k:N - k] for k in range(1, K + 1)], axis=1)  # row n: x[n-1], ..., x[n-K]
    h, *_ = np.linalg.lstsq(X.astype(float), x[K:].astype(float), rcond=None)
    q = np.round(h * 2 ** shift).astype(np.int64)                    # quantized coefficients
    prediction = (X @ q) >> shift                                    # integer arithmetic only
    return q, x[K:] - prediction


def decode_block(warmup: np.ndarray, q: np.ndarray, residual: np.ndarray,
                 shift: int = 12) -> np.ndarray:
    """Invert encode_block: x[n] = e[n] + prediction, one sample at a time (recursive)."""
    x = [int(v) for v in warmup]
    q = [int(c) for c in q]
    for e in residual:
        prediction = sum(c * x[-k] for k, c in enumerate(q, start=1)) >> shift
        x.append(int(e) + prediction)
    return np.array(x)


drums = pq.Audio.from_file("../assets/audio-drums.wav")
x = np.round(drums.samples[:, 0] * 32768).astype(np.int64)  # back to 16-bit integers
B, K = 4096, 8
bits = 0
for start in range(0, len(x) - K, B):
    block = x[start:start + B]
    q, residual = encode_block(block, K)
    k = min(range(16), key=lambda k: rice_bits(zigzag(residual), k))  # best k per block
    bits += 16 * K + 16 * K + 4 + rice_bits(zigzag(residual), k)      # warm-up, coefficients, k, residual
    assert np.array_equal(decode_block(block[:K], q, residual), block)  # lossless!

print(f"Uncompressed: 16 bits per sample")
print(f"Toy FLAC: {bits / len(x):.2f} bits per sample (compression factor {16 * len(x) / bits:.2f}×)")